# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [11]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [12]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 270.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
query_embeddings.shape

(5346, 384)

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [14]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [15]:
import redis

r = redis.Redis(host='localhost', port=6379)

# **Objetivo I**

### **Tarea 1**. Buscar la estructura de datos más apropiada para la cache.

Para esta tarea, la estructura de datos más apropiada para la cache en Redis sería una **tabla Hash**. Esto es ideal para almacenar la información de cada carta, ya que cada carta puede tener múltiples atributos (nombre, texto, tipo, etc.) que pueden ser almacenados como campos dentro del Hash.  Además, los Hashes permiten un acceso rápido a los datos, lo que es crucial para una cache eficiente.


### **Tarea 2**. Escribir una función Python que reciba un fichero .csv con un conjunto de ejemplo de cartas y las cargue en una base de datos Redis usando la estructura de datos seleccionada en el paso anterior.


In [16]:
df = pd.read_csv("cards_final_with_xp.csv")

In [17]:
df.head()

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic


In [18]:
def cargar_cartas_en_redis(df):
    
    for i in range(len(df)):
        
        code = df.loc[i, "code"]
        name = df.loc[i, "name"]
        text = df.loc[i, "text"]
        type_code = df.loc[i, "type_code"]
        traits = df.loc[i, "traits"]
        pack_code = df.loc[i, "pack_code"]
        faction_code = df.loc[i, "faction_code"]
        xp = df.loc[i, "xp"]
        illustrator = df.loc[i, "illustrator"]
        image_url = df.loc[i, "image_url"]
        
        r.hset('carta:' + code, mapping={
            "name": name,
            "text": text,
            "type_code": type_code,
            "traits": traits,
            "pack_code": pack_code,
            "faction_code": faction_code,
            "xp": xp,
            "illustrator": illustrator,
            "image_url": image_url
        })

In [19]:
def cargar_cartas_en_redis(df: pd.DataFrame) -> None:
    """
    Carga las cartas de un DataFrame en Redis.

    Args:
        df (pd.DataFrame): DataFrame que contiene las cartas.
    """
    for _, serie in df.iterrows():

        data = serie.to_dict()
        code = data.pop("code", None)
        
        if code:
            key = f"card:{code}"
            
        r.hset(key, mapping=data)

In [20]:
cargar_cartas_en_redis(df)

### **Tarea 3**. Escribir funciones python que permita realizar cada una de las acciones. Implementar una por acción.

#### 1. Saber si una carta está en la cache por su campo code.

In [21]:
def carta_en_cache(code: str) -> bool:
    """
    Verifica si una carta está en la cache por su campo code.

    Args:
        code (str): El código de la carta a verificar.

    Returns:
        bool: True si la carta está en la cache, False en caso contrario.
    """
    key = f"card:{code}"
    return r.exists(key) == 1

In [22]:
carta_en_cache("01001")

False

In [23]:
carta_en_cache("06095")

True

#### 2. Recuperar todos los datos de una carta a partir de su code.

In [24]:
def recuperar_carta(code: str) -> bool:
    """
    Recupera todos los datos de una carta a partir de su code.
    
    Args:
        code (str): El código de la carta a recuperar.
        
    Returns:
        dict: Un diccionario con los datos de la carta si está en la cache, None en caso contrario.
    """
    key = f"card:{code}"
    return r.hgetall(key) 

In [25]:
recuperar_carta("06095")

{b'xp': b'0',
 b'illustrator': b'Sacha Angel Diener',
 b'text': b'Revelation - Put Deeper Slumber into play in your threat area. Your maximum hand size is reduced by 3 and is checked after each time you draw 1 or more cards. [action] [action]: Discard Deeper Slumber.',
 b'type_code': b'treachery',
 b'image_url': b'https://arkhamdb.com/bundles/cards/06095.jpg',
 b'traits': b'Curse',
 b'pack_code': b'tde',
 b'name': b'Deeper Slumber',
 b'faction_code': b'mythos'}

#### 3. Meter una carta nueva en la cache.


In [26]:
def meter_carta_en_cache(code: str, data: dict) -> None:
    """
    Mete una carta nueva en la cache si no existe ya.

    Args:
        code (str): El código de la carta a meter.
        data (dict): Un diccionario con los datos de la carta a meter.
    """
    key = f"card:{code}"
    r.hset(key, mapping=data) if r.exists(key) == 0 else print(f"La carta con código {code} ya existe en la cache.")

In [27]:
data_example = {
    "name": "Test Card",
    "text": "This is a test card.",
    "type_code": "test_type",
    "traits": "test_traits",
    "pack_code": "test_pack",
    "faction_code": "test_faction",
    "xp": 0,
    "illustrator": "test_illustrator",
    "image_url": "http://example.com/test_card.jpg"
}

In [28]:
meter_carta_en_cache("01001", data_example)

In [29]:
recuperar_carta("01001")

{b'name': b'Test Card',
 b'text': b'This is a test card.',
 b'type_code': b'test_type',
 b'traits': b'test_traits',
 b'pack_code': b'test_pack',
 b'faction_code': b'test_faction',
 b'xp': b'0',
 b'illustrator': b'test_illustrator',
 b'image_url': b'http://example.com/test_card.jpg'}

#### 4. Eliminar una carta de la cache a partir de su campo code.

In [30]:
def eliminar_carta_de_cache(code: str) -> None:
    """
    Elimina una carta de la cache a partir de su campo code.

    Args:
        code (str): El código de la carta a eliminar.
    """
    key = f"card:{code}"
    r.delete(key)
    

In [31]:
eliminar_carta_de_cache("01001")

In [32]:
recuperar_carta("01001")

{}

# **Objetivo II** 

El equipo del portal ha oído hablar de las capacidades de búsqueda avanzada de Redis y quiere probar si se pueden usar para extender la funcionalidad del portal. En concreto han identificado varias búsquedas que han reclamado los usuarios a lo largo de los años:

- **A**. En el juego hay 7 facciones, 5 que son clases que pueden usar los jugadores (mystic, survivor, guardian, seeker y rogue), la facción “neutral” que es equipo común para todas las clases y la facción “mythos” que son los enemigos. Recientemente se han añadido cartas especiales que tienen más de una facción. Los jugadores están interesados en buscar todas las cartas que contengan varias facciones a la vez, así como, buscar todas las cartas que contengan al menos una de las facciones que indiquen. Por defecto devolvemos las cartas de 5 en 5.
- **B**. Los traits son una forma muy cómoda de buscar cartas interesantes y es muy usada por los
usuarios. No obstante, como hay muchos, manejarlos es algo complicado. Los usuarios
quieren saber cuáles son los traits más comunes para su facción por orden de frecuencia.
Por defecto mostramos páginas con 15 traits.

- **C**. Durante el juego los jugadores ganan puntos de experiencia y pueden gastarlos en mejorar sus mazos. Por eso, es normal que los usuarios quieran buscar cartas que contengan ciertos traits y que puedan usarse para actualizar sus cartas, es decir, que tengan un coste de experiencia (xp) > 0. Normalmente están interesados en cartas que sean de su facción, así que quieren que esas aparezcan entre los primeros resultados, no obstante, también quieren ver cartas de otras facciones ya que hay mazos que mezclan varias facciones. No obstante, nunca quieren ver cartas de la facción “mythos” porque no las pueden incluir en su mazo. Los jugadores también piden filtrar las cartas por lo puntos de experiencia que les quedan, para que no les aparezcan cartas que no pueden comparar. Por defecto devolvemos las cartas de 5 en 5.


### Tarea 1. Diseña un indice que permita realizar las búsquedas anteriores.

Para crear este indice, tenemos que tener en cuenta lo siguiente:

- Como vamos a realizar búsquedas por facción, es importante tener un indice que nos permita buscar por este campo, que sera de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varias facciones, es necesario agregar un separador para poder almacenar varias facciones en el mismo campo. En este caso, tendremos que usar el separador “|” para separar las facciones en el campo de facciones. 

- Para la búsqueda por traits, es importante tener un indice que nos permita buscar por este campo, que sera de tipo TAG debido a que los valores son cadenas de texto concretas. Ademas de esto, como las cartas pueden tener varios traits, es necesario agregar un separador para poder almacenar varios traits en el mismo campo. En este caso, tendremos que usar el separador “|” para separar los traits en el campo de traits.

- Para la búsqueda por coste de experiencia, es importante tener un indice que nos permita buscar por este campo, que sera de tipo NUMERIC debido a que los valores son números enteros. Ademas de esto, como queremos ordenarlos por coste de experiencia, es necesario agregar un SORTABLE para poder ordenar los resultados por este campo.

In [33]:
from redis.exceptions import ResponseError

def crear_indice(nombre_indice: str) -> None:
    """
    Crea un indice en Redis para las cartas.
    """

    # TODO: Usar este o el comando equivalente para crear el indice, teniendo en cuenta los campos que queremos indexar y el tipo de cada campo.
    """
    r.ft(nombre_indice).create_index(
        [
            redis.commands.search.field.TagField("faction_code", separator="|"),
            redis.commands.search.field.TagField("traits", separator="|"),
            redis.commands.search.field.NumericField("xp", sortable=True)
        ]
    )
    """
    try:
        comando = f"""
        FT.CREATE {nombre_indice} 
        ON HASH PREFIX 1 card: 
        SCHEMA faction_code TAG SEPARATOR | 
        traits TAG SEPARATOR | 
        xp NUMERIC SORTABLE
        """
        
        r.execute_command(*comando.split())
        print(f"Índice '{nombre_indice}' creado con éxito.")
        
    except ResponseError as e:
        if "Index already exists" in str(e):
            print(f"El índice '{nombre_indice}' ya existe. No es necesario crearlo de nuevo.")
        else:
            print(f"Error de Redis al crear el índice: {e}")
            
    except Exception as e:
        print(f"Error inesperado: {e}")


In [34]:
crear_indice("cards-idx")

Índice 'cards-idx' creado con éxito.


Para poder hacer:
* A) AND/OR de facciones (incluye cartas multifacción) → campo faction_code como TAG con SEPARATOR |.
* B) “traits más comunes por facción” → necesito poder agregar y contar (FT.AGGREGATE: GROUPBY/REDUCE COUNT).
* C) filtrar por xp > 0 y por xp máximo, excluir mythos y paginar → xp como NUMERIC (rangos) y si quiero ordenar, hacerlo SORTABLE.

1) Índice (FT.CREATE)
Dado que las cartas están en HASH con un prefijo card: (ej.: card:1029) y que dentro del hash existen los campos del enunciado.

In [35]:
import redis
from redis.exceptions import ResponseError

def init_redis(host="localhost", port=6379, db=0) -> redis.Redis:
    # decode_responses=True para trabajar con strings (más cómodo en notebooks)
    return redis.Redis(host=host, port=port, db=db, decode_responses=True)

def ensure_cards_index(r: redis.Redis, index_name="cards-idx", prefix="card:", drop=False) -> None:
    """
    Crea el índice si no existe.
    - faction_code y traits son TAG multi-valor separados por '|'
    - xp es NUMERIC y SORTABLE para filtrar y (opcionalmente) ordenar.
    """
    if drop:
        try:
            r.execute_command("FT.DROPINDEX", index_name, "DD")
        except ResponseError:
            pass

    # Si existe, no hacemos nada
    try:
        r.execute_command("FT.INFO", index_name)
        return
    except ResponseError:
        pass

    # FT.CREATE ON HASH PREFIX ... SCHEMA ...
    # Sintaxis base: :contentReference[oaicite:4]{index=4}
    r.execute_command(
        "FT.CREATE", index_name,
        "ON", "HASH",
        "PREFIX", "1", prefix,
        "SCHEMA",
        # Campos clave para Objetivo II
        "code", "TAG",
        "name", "TEXT",
        "text", "TEXT",
        "type_code", "TAG",
        "pack_code", "TAG",
        "faction_code", "TAG", "SEPARATOR", "|",   # multi-facción 
        "traits", "TAG", "SEPARATOR", "|",         # multi-traits 
        "xp", "NUMERIC", "SORTABLE",               # rangos/ordenación 
        # Campos extra (no imprescindibles para Obj II, pero útiles para RETURN)
        "illustrator", "TEXT",
        "image_url", "TEXT"
    )


* Usamos TAG SEPARATOR | porque el enunciado dice que traits y faction_code vienen separados por | si hay más de uno, y los PDFs explican que TAG multi-valor se soporta con SEPARATOR.
* FT.SEARCH soporta paginación con LIMIT offset n y devolver campos con RETURN.

### Tarea 2. Implementa una función python que realize cada una de las búsquedas anteriores

#### Helpers para parsear respuestas

In [36]:
def _parse_ft_search(raw):
    """
    Parse simple para FT.SEARCH (RESP2):
    [total, key1, [field, value, field, value...], key2, [...], ...]
    """
    if raw is None:
        return 0, []
    if isinstance(raw, dict):
        # Algunas configs RESP3 pueden devolver dict; aquí lo dejamos por si acaso
        total = raw.get("total", 0)
        docs = raw.get("documents", [])
        return total, docs

    total = int(raw[0])
    out = []
    i = 1
    while i < len(raw):
        key = raw[i]
        fields = raw[i + 1]
        doc = {"_key": key}
        # fields es lista [k,v,k,v,...]
        for j in range(0, len(fields), 2):
            doc[fields[j]] = fields[j + 1]
        out.append(doc)
        i += 2
    return total, out

def _parse_ft_aggregate(raw):
    """
    Parse simple para FT.AGGREGATE (RESP2):
    [total, [k,v,k,v...], [k,v,k,v...], ...]
    """
    if raw is None:
        return 0, []
    if isinstance(raw, dict):
        total = raw.get("total", 0)
        rows = raw.get("results", [])
        return total, rows

    total = int(raw[0])
    rows = []
    for row in raw[1:]:
        d = {}
        for j in range(0, len(row), 2):
            d[row[j]] = row[j + 1]
        rows.append(d)
    return total, rows


#### **Objetivo II-A**: búsqueda por facciones

In [66]:
def search_by_factions(
    r: redis.Redis,
    factions: list[str],
    mode: str = "OR",            # "OR" o "AND"
    page: int = 0,
    page_size: int = 5,
    index_name: str = "cards-idx",
):
    """
    Devuelve cartas que:
    - OR: tengan al menos una de las facciones
    - AND: tengan todas las facciones (para multifacción)
    Paginado de 5 en 5 (por defecto).
    """
    if not factions:
        raise ValueError("factions no puede estar vacío")

    factions = [f.strip().lower() for f in factions]

    if mode.upper() == "AND":
        # AND = espacios (intersección): "@faction:{A} @faction:{B}"
        query = " ".join([f"@faction_code:{{{f}}}" for f in factions])
    else:
        # OR dentro del TAG: "@faction:{A|B|C}"
        query = f"@faction_code:{{{'|'.join(factions)}}}"

    offset = page * page_size

    raw = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "LIMIT", str(offset), str(page_size)
    )
    total, docs = _parse_ft_search(raw)
    return {"total": total, "page": page, "page_size": page_size, "results": docs, "query": query}


In [ ]:
def search_by_factions(
    r: redis.Redis,
    factions: list[str],
    mode: str = "OR",            # "OR" o "AND"
    page: int = 0,
    page_size: int = 5,
    index_name: str = "cards-idx",
):
    """
    Devuelve cartas MULTIFACCIÓN (estrictamente más de una facción) que:
    - OR: tengan al menos una de las facciones y al menos otra diferente.
    - AND: tengan todas las facciones (si es solo una, obliga a tener otra).
    Paginado de 5 en 5 (por defecto).
    """
    if not factions:
        raise ValueError("factions no puede estar vacío")

    factions = [f.strip().lower() for f in factions]
    
    # Facciones válidas para multifacción (se excluyen 'neutral' y 'mythos')
    valid_factions = ["mystic", "survivor", "guardian", "seeker", "rogue"]

    if mode.upper() == "AND":
        if len(factions) == 1:
            # Si piden 1 sola, exigimos que tenga esa AND alguna de las demás válidas
            fac = factions[0]
            others = [f for f in valid_factions if f != fac]
            query = f"@faction_code:{{{fac}}} @faction_code:{{{'|'.join(others)}}}"
        else:
            # Si piden 2 o más con AND, ya es obligatoriamente multifacción por definición
            query = " ".join([f"@faction_code:{{{f}}}" for f in factions])
    else:
        # Modo OR: Para cada facción pedida, construimos la condición de que sea multifacción
        or_blocks = []
        for fac in factions:
            others = [f for f in valid_factions if f != fac]
            # Bloque: (Tiene la facción actual AND tiene al menos una de las otras)
            block = f"(@faction_code:{{{fac}}} @faction_code:{{{'|'.join(others)}}})"
            or_blocks.append(block)
        
        # Unimos todos los bloques con OR lógico a nivel de query
        query = " | ".join(or_blocks)

    offset = page * page_size

    # Ejecutamos el comando manteniendo tu RETURN y LIMIT
    raw = r.execute_command(
        "FT.SEARCH", index_name, query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "LIMIT", str(offset), str(page_size)
    )
    
    # Asumo que _parse_ft_search ya la tienes implementada por tu cuenta
    total, docs = _parse_ft_search(raw)
    
    return {
        "total": total, 
        "page": page, 
        "page_size": page_size, 
        "results": docs, 
        "query": query
    }

In [68]:
search_by_factions(r, ["guardian", "seeker"], mode="or", page=41, page_size=5)

{'total': 509,
 'page': 41,
 'page_size': 5,
 'results': [{'_key': 'card:4230',
   'name': 'Ancient Stone',
   'faction_code': 'seeker',
   'xp': '4'},
  {'_key': 'card:7018',
   'name': 'Blessed Blade',
   'faction_code': 'guardian',
   'xp': '0'},
  {'_key': 'card:10043',
   'name': 'Mouse Mask',
   'faction_code': 'seeker',
   'xp': '0'},
  {'_key': 'card:2260',
   'name': 'Leadership',
   'faction_code': 'guardian',
   'xp': '0'},
  {'_key': 'card:08037',
   'name': 'Survey the Area',
   'faction_code': 'seeker',
   'xp': '1'}],
 'query': '@faction_code:{guardian|seeker}'}

### Objetivo II-B: traits más comunes por facción

4) Objetivo II-B: traits más comunes por facción (frecuencia), páginas de 15
Aquí lo natural es FT.AGGREGATE con GROUPBY + REDUCE COUNT, tal como aparece en el PDF de agregaciones.
Como traits viene “aplanado” en un string con |, hay 2 enfoques:
B1) (Preferido) FT.AGGREGATE con split + UNWIND (si tu RediSearch lo soporta)

In [41]:
from collections import Counter

def _top_traits_fallback_python(r, faction, page, page_size, index_name):
    query = f"@faction_code:{{{faction}}}"
    # Pedimos solo traits para TODAS las cartas de esa facción (en lotes)
    # Truco: LIMIT 0 0 devuelve solo el nº de aciertos :contentReference[oaicite:12]{index=12}
    raw_count = r.execute_command("FT.SEARCH", index_name, query, "LIMIT", "0", "0")
    total_cards = int(raw_count[0])

    counter = Counter()
    batch = 500
    for start in range(0, total_cards, batch):
        raw = r.execute_command(
            "FT.SEARCH", index_name, query,
            "RETURN", "1", "traits",
            "LIMIT", str(start), str(batch)
        )
        _, docs = _parse_ft_search(raw)
        for d in docs:
            traits = (d.get("traits") or "").strip()
            if not traits:
                continue
            for t in traits.split("|"):
                t = t.strip()
                if t:
                    counter[t] += 1

    # Orden por frecuencia desc
    items = sorted(counter.items(), key=lambda x: (-x[1], x[0]))

    offset = page * page_size
    return {
        "total_traits": len(items),
        "page": page,
        "page_size": page_size,
        "results": items[offset: offset + page_size],
        "method": "FT.SEARCH + Python Counter (fallback)"
    }


In [42]:
def top_traits_for_faction(
    r: redis.Redis,
    faction: str,
    page: int = 0,
    page_size: int = 15,
    index_name: str = "cards-idx",
):
    """
    Devuelve los traits más comunes para una facción (orden freq desc), paginado 15.
    Intenta hacerlo con FT.AGGREGATE (GROUPBY/COUNT).
    """
    faction = faction.strip().lower()
    offset = page * page_size

    # Query: solo cartas de esa facción
    query = f"@faction_code:{{{faction}}}"

    # OJO: split/unwind depende de versión. Si falla, hacemos fallback (B2).
    try:
        raw = r.execute_command(
            "FT.AGGREGATE", index_name, query,
            "LOAD", "1", "traits",
            "APPLY", "split(@traits, '|')", "AS", "trait",
            "GROUPBY", "1", "@trait",
            "REDUCE", "COUNT", "0", "AS", "frecuencia",
            "SORTBY", "2", "@frecuencia", "DESC",
            "LIMIT", str(offset), str(page_size)
            )
        total, rows = _parse_ft_aggregate(raw)
        # Normalizo salida: lista de (trait, freq)
        out = [(row.get("trait"), int(row.get("frecuencia", 0))) for row in rows if row.get("trait")]
        return {"total_traits": total, "page": page, "page_size": page_size, "results": out, "method": "FT.AGGREGATE"}
    except ResponseError:
        return _top_traits_fallback_python(r, faction, page, page_size, index_name)


In [43]:
def top_traits_for_faction(
    r,
    faction: str,
    page: int = 0,
    page_size: int = 15,
    index_name: str = "cards-idx",
):
    faction = faction.strip().lower()
    offset = page * page_size

    query = f"@faction_code:{{{faction}}}"

    try:
        raw = r.execute_command(
            "FT.AGGREGATE", index_name, query,
            "LOAD", "1", "traits",
            "APPLY", "split(@traits, '|')", "AS", "trait",
            "GROUPBY", "1", "@trait",
            "REDUCE", "COUNT", "0", "AS", "frecuencia",
            "SORTBY", "2", "@frecuencia", "DESC",
            "LIMIT", str(offset), str(page_size)
        )

        total_groups = raw[0]
        results = []

        for i in range(1, len(raw)):
            row_data = raw[i]
            
            # NUEVO: Decodificamos claves y valores si vienen como bytes
            res_dict = {}
            for j in range(0, len(row_data), 2):
                key = row_data[j].decode('utf-8') if isinstance(row_data[j], bytes) else row_data[j]
                val = row_data[j+1].decode('utf-8') if isinstance(row_data[j+1], bytes) else row_data[j+1]
                res_dict[key] = val
            
            t_name = res_dict.get("trait")
            
            if t_name and t_name.strip():
                results.append((t_name.strip(), int(res_dict.get("frecuencia", 0))))

        return {
            "faction": faction,
            "total_traits": total_groups,
            "page": page,
            "page_size": page_size,
            "results": results,
            "method": "FT.AGGREGATE"
        }

    except Exception as e:
        print(f"Error ejecutando FT.AGGREGATE: {e}")
        return None

In [44]:
top_traits_for_faction(r, "survivor", page=0, page_size=15)

{'faction': 'survivor',
 'total_traits': 57,
 'page': 0,
 'page_size': 15,
 'results': [('Item', 36),
  ('Innate', 16),
  ('Ally', 14),
  ('Talent', 14),
  ('Tool', 13),
  ('Spirit', 13),
  ('Weapon', 12),
  ('Fortune', 12),
  ('Melee', 11),
  ('Blessed', 11),
  ('Trick', 9),
  ('Tactic', 8),
  ('Charm', 8),
  ('Improvised', 8),
  ('Developed', 6)],
 'method': 'FT.AGGREGATE'}

B2) (Fallback robusto) FT.SEARCH + contar en Python
Sigue usando RediSearch para filtrar por facción, y luego agregas tú. Si el dataset es el del curso, suele ser manejable.

5) Objetivo II-C: buscar cartas con xp>0, excluir mythos, filtrar por xp máximo, priorizar facción y permitir mezcla
El enunciado pide:
xp > 0
excluir mythos siempre
filtrar por xp máximo
mostrar “preferidas” de una facción al principio, pero permitir otras facciones después
páginas de 5 
practica_redis_busqueda_hibrida…
Para que sea simple y 100% controlable, lo hago en 2 consultas:
primero facción preferida
luego el resto (sin duplicados)
y mezclo manteniendo el orden “preferida primero”.

In [45]:
def search_upgrades(
    r: redis.Redis,
    traits: list[str] | None = None,          # traits deseados (OR)
    preferred_faction: str | None = None,     # facción a priorizar
    allowed_factions: list[str] | None = None,# facciones permitidas (mezcla); si None => todas excepto mythos
    xp_max: int | None = None,                # filtro xp <= xp_max
    page: int = 0,
    page_size: int = 5,
    index_name: str = "cards-idx",
):
    """
    C) Cartas "upgrades":
      - xp > 0
      - xp <= xp_max (si se da)
      - excluir mythos
      - (opcional) filtrar por traits
      - priorizar preferred_faction en los primeros resultados
      - paginado 5
    """
    # ---- filtros base ----
    parts = []

    # xp > 0 y opcional xp_max (rangos NUMERIC) :contentReference[oaicite:14]{index=14}
    if xp_max is None:
        parts.append("@xp:[(1 +inf]")
    else:
        parts.append(f"@xp:[(1 {int(xp_max)}]")

    # excluir mythos (negación en TAG) 
    parts.append("-@faction_code:{mythos}")

    # filtrar por traits (OR)
    if traits:
        traits = [t.strip() for t in traits if t.strip()]
        if traits:
            parts.append(f"@traits:{{{'|'.join(traits)}}}")

    # restringir a ciertas facciones si el usuario quiere “mezclar” solo algunas
    if allowed_factions:
        allowed_factions = [f.strip().lower() for f in allowed_factions]
        parts.append(f"(@faction_code:{{{'|'.join(allowed_factions)}}})")

    base_query = " ".join(parts) if parts else "*"

    # ---- función para lanzar FT.SEARCH con orden estable ----
    def _search(query, limit_n):
        raw = r.execute_command(
            "FT.SEARCH", index_name, query,
            "RETURN", "4", "code", "name", "faction_code", "xp",
            "SORTBY", "xp", "ASC",   # opcional: muestra primero lo “más barato” si es SORTABLE :contentReference[oaicite:16]{index=16}
            "LIMIT", "0", str(limit_n),
        )
        _, docs = _parse_ft_search(raw)
        return docs

    needed = (page + 1) * page_size  # pedimos suficiente para cubrir hasta esa página

    results = []
    seen = set()

    # 1) primero preferred_faction (si existe)
    if preferred_faction:
        pf = preferred_faction.strip().lower()
        q_pref = f"{base_query} @faction_code:{{{pf}}}"
        pref_docs = _search(q_pref, needed)
        for d in pref_docs:
            code = d.get("code")
            if code and code not in seen:
                seen.add(code)
                results.append(d)

    # 2) luego resto (sin duplicar)
    # Si hay preferred_faction, excluimos esa facción aquí para que no repita
    if preferred_faction:
        pf = preferred_faction.strip().lower()
        q_other = f"{base_query} -@faction_code:{{{pf}}}"
    else:
        q_other = base_query

    other_docs = _search(q_other, needed)
    for d in other_docs:
        code = d.get("code")
        if code and code not in seen:
            seen.add(code)
            results.append(d)

    # paginación final
    start = page * page_size
    page_docs = results[start: start + page_size]

    return {
        "page": page,
        "page_size": page_size,
        "returned": len(page_docs),
        "query_base": base_query,
        "preferred_faction": preferred_faction,
        "results": page_docs
    }


In [46]:
def search_upgrades(
    r,
    traits=None,
    preferred_faction=None,
    allowed_factions=None,
    xp_max=None,
    page=0,
    page_size=5,
    index_name="cards-idx",
):
    query = []

    if xp_max is None:
        query.append("@xp:[(1 +inf]")
    else:
        query.append(f"@xp:[(1 {int(xp_max)}]")

    # excluir mythos (negación en TAG) 
    query.append("-@faction_code:{mythos}")


    # filtrar por traits (OR)
    if traits:
        traits = [t.strip() for t in traits if t.strip()]
        if traits:
            query.append(f"@traits:{{{'|'.join(traits)}}}")

    # restringir a ciertas facciones si el usuario quiere “mezclar” solo algunas
    if allowed_factions:
        allowed_factions = [f.strip().lower() for f in allowed_factions]
        query.append(f"(@faction_code:{{{'|'.join(allowed_factions)}}})")

    
    query.append(f"~@faction_code:{preferred_faction}" if preferred_faction else "")

    base_query = " ".join(query) if query else "*"

    raw_result = r.execute_command(
        "FT.SEARCH", index_name, base_query,
        "RETURN", "4", "code", "name", "faction_code", "xp",
        "SORTBY", "xp", "ASC",
        "LIMIT", str(page * page_size), str(page_size)
    )

    total_cards = raw_result[0] if raw_result else 0
    parsed_results = []
    

    for i in range(1, len(raw_result), 2):
        fields_list = raw_result[i + 1]
        
        doc = dict(zip(fields_list[0::2], fields_list[1::2]))
        
        parsed_results.append((doc.get('name', ''), doc.get('xp', '0')))

    return {
        'faction': preferred_faction,
        'total_cards': total_cards,
        'page': page,
        'page_size': page_size,
        'results': parsed_results,
        'method': 'FT.SEARCH'
    }

In [47]:
search_upgrades(
    r,
    traits=["Item"],
    preferred_faction="guardian",
    allowed_factions=["guardian", "survivor"],  # ejemplo de mezcla
    xp_max=7,
    page=0,
    page_size=5
)

{'faction': 'guardian',
 'total_cards': 445,
 'page': 0,
 'page_size': 5,
 'results': [('', '0'), ('', '0'), ('', '0'), ('', '0'), ('', '0')],
 'method': 'FT.SEARCH'}

6) ¿Cómo comprobar que está bien implementado? (lo que te piden en el punto 3)
Te dejo un checklist “tipo entrega”:
Verificación A (facciones AND/OR)
Elige 2 facciones (ej.: ["mystic","guardian"]).
Ejecuta:
AND: todas las cartas devueltas deben tener ambas en faction_code
OR: todas deben tener al menos una
Puedes validarlo con un assert simple:

In [48]:
def check_A(results, factions, mode):
    factions = set(f.lower() for f in factions)
    for d in results["results"]:
        card_factions = set((d.get("faction_code") or "").split("|"))
        if mode == "AND":
            assert factions.issubset(card_factions)
        else:
            assert len(factions.intersection(card_factions)) > 0


Verificación B (traits por facción)
Coge una facción (ej. mystic).
Obtén top_traits_for_faction(...) para la primera página.
Valida la frecuencia comparando:
(i) tu resultado
(ii) un conteo manual en Python para esa facción (el propio fallback ya te hace esa validación indirecta).
Además, si usas FT.AGGREGATE, estás usando exactamente el mecanismo de “GROUP BY + COUNT” explicado en los PDFs.
Verificación C (xp>0, excluir mythos, xp_max y prioridad)
Para una consulta C concreta, verifica:
xp de cada resultado: int(xp) > 0 y <= xp_max si existe (rangos NUMERIC).
faction_code no contiene mythos (negación).
Si pones preferred_faction="seeker", los primeros resultados (antes de “rellenar”) pertenecen a esa facción (tu mezcla en 2 pasos lo garantiza).
Y un truco útil del temario: FT.SEARCH ... LIMIT 0 0 para comprobar el número total de aciertos sin traer documentos (te sirve para comparar “cuántas” devuelve tu filtro).

Ejemplo de uso

In [49]:
r = init_redis(host="localhost", port=6379)
ensure_cards_index(r)

# A) OR
res_or = search_by_factions(r, ["mystic", "guardian"], mode="OR", page=0)
# A) AND (multifacción)
res_and = search_by_factions(r, ["mystic", "guardian"], mode="AND", page=0)

# B) Top traits mystic (15)
top_traits = top_traits_for_faction(r, "mystic", page=0)

# C) Upgrades: xp>0, xp<=3, excluir mythos, preferir seeker, mezclar seeker+rogue
upg = search_upgrades(
    r,
    traits=["Weapon", "Spell"],       # ejemplo: OR
    preferred_faction="seeker",
    allowed_factions=["seeker", "rogue"],
    xp_max=3,
    page=0
)

# **Objetivo III**

El equipo del portal quiere implementar un recomendador automático de cartas. La idea es que, dada una carta, el sistema muestre otras 5 cartas parecidas a dicha carta que a los usuarios les pudieran interesar. Para ello, se ha consultado con el equipo de diseño de “Arkham Dread” y han dicho que lo mejor sería utilizar el nombre y el texto de la carta, si no fuera posible también indican que los traits son una alternativa, aunque son menos específicos generales. Al preguntarles por las ilustraciones, el equipo de diseño ha respondido que están más pensadas como decoración y no dan información sobre lo que hace la carta. Con esta información el equipo de desarrollo ha planteado 4 alternativas de más sencilla a más compleja:
- **A.** Usar solo metadatos de la carta y obviar el nombre y el texto: Proponen buscar cartas de
la misma facción y que tengan alguno de los traits de la carta sobre la que se realiza la
búsqueda.
- **B.** Capacidades full-text de redis: Aprovechar las capacidades de redis y hace una busqueda
full-text con el texto de la carta.
- **C.** Búsqueda semántica: Hacer embeddings con el nombre y el texto de la carta para luego
hacer búsqueda semántica.


### Tarea 1. Diseña un índice que permita realizar las alternativas anteriores.